# 🥇 XAUUSD Neural Lab - Advanced Bi-LSTM Training

Jupyter Notebook ini didesain untuk dijalankan di **Kaggle** atau **Google Colab**.
Model ini adalah arsitektur **3-Layer Bidirectional LSTM** yang akan mempelajari pola pergerakan harga Emas (XAUUSD) digali dari 22+ *Technical Features*, serta data korelasi makro-ekonomi seperti **DXY (Dollar Index)** dan **US10Y (Bond Yields)**.

### 🚀 Cara Penggunaan:
1. Jalankan semua cell secara berurutan.
2. Di bagian Dataset, pastikan Anda telah mengunggah file CSV historis XAUUSD yang berisi OHLCV, DXY, US10Y, dsb.
3. Setelah *training* selesai, script ini akan menghasilkan file `lstm_weights_v2.json`.
4. Copy isi file JSON tersebut dan gantikan *existing weights* di web app (file `src/lib/lstm-weights.ts`).

In [ ]:
!pip install yfinance pandas-ta "numpy<2.2.0" "pandas==2.2.2" -q
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import pandas_ta as ta
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import json
import datetime
import os
import yfinance as yf

print(f"TensorFlow Version: {tf.__version__}")


## 1. Load & Preprocess Dataset
Diasumsikan Anda memiliki file `xauusd_merged_1h.csv` dengan kolom OHLCV murni ditambah:
`dxy_close` (USD Index), `us10y_close` (Bond Yield 10 Year), `cot_index` (Commitment of Traders), `is_nfp_week` (Non-Farm Payroll Flag).

In [ ]:
def fetch_real_data(period='730d', interval='1h'):
    print(f'Mengunduh data {period} {interval} Live dari Yahoo Finance...')
    gold = yf.download('GC=F', period=period, interval=interval, progress=False)
    if isinstance(gold.columns, pd.MultiIndex):
        gold.columns = gold.columns.droplevel(1)
    
    df = pd.DataFrame()
    df['open'] = gold['Open']
    df['high'] = gold['High']
    df['low'] = gold['Low']
    df['close'] = gold['Close']
    df['volume'] = gold['Volume'] if 'Volume' in gold else 1
    df.fillna(method='ffill', inplace=True)
    df.dropna(inplace=True)
    df = df.reset_index()
    if 'Datetime' in df.columns:
        df.rename(columns={'Datetime': 'time'}, inplace=True)
    elif 'index' in df.columns:
        df.rename(columns={'index': 'time'}, inplace=True)
    return df

df = fetch_real_data()
print("Dataset Asli XAUUSD (GC=F):", df.shape)
df.head()


## 2. Feature Engineering (Technical + Macro)
Menghitung The 22-Feature Extractor ditambah MACRO parameters.

In [ ]:
def add_10_features(df):
    df_copy = df.copy()
    df_copy['ema_20'] = ta.ema(df_copy['close'], length=20)
    df_copy['ema_50'] = ta.ema(df_copy['close'], length=50)
    df_copy['rsi_14'] = ta.rsi(df_copy['close'], length=14)
    
    df_copy['tp'] = (df_copy['high'] + df_copy['low'] + df_copy['close']) / 3
    df_copy['vwap'] = (df_copy['tp'] * df_copy['volume']).cumsum() / df_copy['volume'].cumsum()
    atr = ta.atr(df_copy['high'], df_copy['low'], df_copy['close'], length=14)
    
    df_copy['norm_close'] = (df_copy['close'] - df_copy['ema_20']) / (df_copy['ema_20'] * 0.01)
    df_copy['rsi'] = df_copy['rsi_14'] / 100.0
    df_copy['vwap_dist'] = (df_copy['close'] - df_copy['vwap']) / (df_copy['vwap'] * 0.01)
    df_copy['atr_ratio'] = (atr / df_copy['close']) * 100.0
    df_copy['momentum'] = df_copy['close'].pct_change(periods=10) * 100.0
    df_copy['ema_cross'] = (df_copy['ema_20'] - df_copy['ema_50']) / (df_copy['ema_50'] * 0.01)
    
    vol_mean = df_copy['volume'].mean()
    vol_std = df_copy['volume'].std() if df_copy['volume'].std() != 0 else 1
    df_copy['vol_z'] = (df_copy['volume'] - vol_mean) / vol_std
    
    df_copy['body_ratio'] = abs(df_copy['close'] - df_copy['open']) / (df_copy['high'] - df_copy['low'] + 1e-6)
    hours = df_copy['time'].dt.hour if pd.api.types.is_datetime64_any_dtype(df_copy['time']) else pd.to_datetime(df_copy['time']).dt.hour
    df_copy['hour_sin'] = np.sin(2 * np.pi * hours / 24)
    df_copy['hour_cos'] = np.cos(2 * np.pi * hours / 24)
    
    future_return = df_copy['close'].shift(-1) / df_copy['close'] - 1
    threshold = 0.0005 # 0.05% pergerakan per jam
    conditions = [(future_return > threshold), (future_return < -threshold)]
    choices = [0, 1] # 0 = UP, 1 = DOWN, 2 = NEUTRAL
    df_copy['target'] = np.select(conditions, choices, default=2)
    
    df_copy.dropna(inplace=True)
    return df_copy

df_features = add_10_features(df)
print("Distribusi label (UP=0, DOWN=1, NEUT=2):\n", df_features['target'].value_counts(normalize=True))


## 3. Create Sequences for LSTM
LSTM membutuhkan data tiga dimensi: `[samples, lookback, features]`

In [ ]:
LOOKBACK = 60

feature_cols = ['norm_close', 'rsi', 'vwap_dist', 'atr_ratio', 'momentum', 'ema_cross', 'vol_z', 'body_ratio', 'hour_sin', 'hour_cos']
print(f"Menggunakan {len(feature_cols)} Input Features")

scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_features[feature_cols])
target_series = df_features['target'].values
targets = pd.get_dummies(target_series).values

X, y, y_labels = [], [], []
for i in range(LOOKBACK, len(scaled_data)):
    X.append(scaled_data[i-LOOKBACK:i])
    y.append(targets[i])
    y_labels.append(target_series[i])

X, y = np.array(X), np.array(y)

# Hitung class weights otomatis untuk menyeimbangkan UP, DOWN, NEUTRAL
cw = compute_class_weight('balanced', classes=np.unique(y_labels), y=y_labels)
class_weights = {i: weight for i, weight in enumerate(cw)}
print("Class Weights:", class_weights)

split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
print(f"X_train shape: {X_train.shape}")


## 4. Build Bi-LSTM Architecture
Arsitektur VVIP: 3-Layer Bidirectional LSTM dengan Dropout untuk mencegah Overfitting.

In [ ]:
!pip install keras-tuner -q
import keras_tuner as kt

def model_builder(hp):
    model = Sequential()
    
    # Tuning ukuran Otak 1
    hp_units_1 = hp.Int('units_1', min_value=64, max_value=256, step=64)
    model.add(Bidirectional(LSTM(hp_units_1, return_sequences=True), input_shape=(LOOKBACK, len(feature_cols))))
    model.add(Dropout(hp.Float('dropout_1', min_value=0.1, max_value=0.4, step=0.1)))
    
    # Tuning ukuran Otak 2
    hp_units_2 = hp.Int('units_2', min_value=32, max_value=128, step=32)
    model.add(Bidirectional(LSTM(hp_units_2, return_sequences=True)))
    model.add(Dropout(hp.Float('dropout_2', min_value=0.1, max_value=0.4, step=0.1)))
    
    # Tuning ukuran Otak 3
    hp_units_3 = hp.Int('units_3', min_value=16, max_value=64, step=16)
    model.add(Bidirectional(LSTM(hp_units_3, return_sequences=False)))
    model.add(Dropout(hp.Float('dropout_3', min_value=0.1, max_value=0.3, step=0.1)))
    
    # Layer Akhir
    hp_dense = hp.Int('dense_units', min_value=32, max_value=128, step=32)
    model.add(Dense(hp_dense, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(3, activation='softmax'))
    
    # Auto-Pilih kecepatan belajar (Learning Rate)
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 5e-4])
    
    model.compile(optimizer=Adam(learning_rate=hp_learning_rate),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model

tuner = kt.Hyperband(model_builder,
                     objective='val_accuracy',
                     max_epochs=50,
                     factor=3,
                     directory='keras_tuner_dir',
                     project_name='xauusd_bilstm_tuner')

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

print("🚀 Memulai AUTO-TUNING: Mesin Kaggle sedang menguji puluhan arsitektur...")
tuner.search(X_train, y_train, epochs=50, validation_data=(X_test, y_test), callbacks=[early_stop], class_weight=class_weights)

# Mesin mengambil kombinasi dengan Akurasi TERTINGGI
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print(f"\n✅ KOMBINASI TERBAIK BERHASIL DITEMUKAN!")
print(f"Layer 1: {best_hps.get('units_1')} unit, Layer 2: {best_hps.get('units_2')} unit, Layer 3: {best_hps.get('units_3')} unit")

print("\n🚀 MELAKUKAN FINAL TRAINING DENGAN SETTINGAN TERBAIK...")
model = tuner.hypermodel.build(best_hps)
history = model.fit(X_train, y_train, epochs=200, batch_size=64, validation_data=(X_test, y_test), callbacks=[early_stop], class_weight=class_weights)


## 5. Training Phase

In [ ]:
# Pelatihan sudah selesai di atas oleh Keras Tuner


## 6. Evaluation

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss Over Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title('Accuracy Over Epochs')
plt.legend()
plt.show()

loss, acc = model.evaluate(X_test, y_test)
print(f"\nTest Accuracy: {acc*100:.2f}%")

## 7. Export Weights for Next.js (TypeScript)
Model LSTM JavaScript di Arra7 membaca model dalam bentuk matriks array.

In [ ]:
def export_weights_to_json(model, filename='lstm_weights_v2.json'):
    weights_dict = {}
    for layer in model.layers:
        layer_weights = layer.get_weights()
        if len(layer_weights) > 0:
            # Convert numpy arrays to lists
            weights_dict[layer.name] = [w.tolist() for w in layer_weights]
    
    # Tambahkan Metadata
    total_params = model.count_params()
    export_data = {
        'metadata': {
            'architecture': 'Bi-LSTM 3-Layer (VVIP Edition)',
            'biLstmUnits': [128, 64, 32],
            'denseUnits': [64, 3],
            'totalParams': total_params,
            'accuracy': float(acc),
            'trainedAt': datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S UTC"),
            'lookback': LOOKBACK,
            'epochs': len(history.history['loss']),
            'features': len(feature_cols)
        },
        'weights': weights_dict
    }
    
    with open(filename, 'w') as f:
        json.dump(export_data, f)
        
    print(f"✅ Weights successfully exported to {filename}")
    print(f"File size: {os.path.getsize(filename) / (1024*1024):.2f} MB")

export_weights_to_json(model)